# 04 — Model training and evaluation

**Owner 3. This uses Module 3 and Module 4.**

The job is multiclass classification: given the conditions around a collision, predict **Fatal / Serious / Slight**.

The dataset only contains collisions. We cannot predict how likely a crash is. We predict how severe it would be, then the app turns those probabilities into a relative safety score.

All reusable code is in `ml/src/train.py`. This notebook is the evidence: dummy baseline first, then Logistic Regression fully evaluated, then two comparison models, then a written choice.

The production artifact is the **whole sklearn Pipeline** (scaler + one-hot encoder + model) saved as `model.joblib`. That is what stops training/serving skew.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, train

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

## 1. Load the feature table

Owner 2 delivers `collisions_features.parquet`. If this clone does not have it (the file is gitignored), `prepare_feature_table()` rebuilds it with **the same** `clean()` and `build_features()` functions already owned by Owners 1 and 2.

The model columns are the lists in `features.get_feature_columns()`:

- **Categorical** — one-hot encoded, because the STATS19 codes are labels, not quantities.
- **Numeric** — standardised, because Logistic Regression is sensitive to scale.

`grid_risk` is **not** a model feature. A cell's severity index includes the collision being predicted, which would leak the target.

In [2]:
df = train.prepare_feature_table()
X, y, feature_columns, cat_cols, num_cols = train.xy_from_features(df)

print("Rows:", f"{len(X):,}")
print("Categorical:", cat_cols)
print("Numeric:", num_cols)
print()
print(y.value_counts().sort_index().rename(config.SEVERITY_LABELS))
print()
print((y.value_counts(normalize=True).sort_index() * 100).round(2).rename(config.SEVERITY_LABELS))

Rows: 513,698
Categorical: ['time_of_day', 'is_weekend', 'road_type', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'junction_detail', 'adverse_conditions', 'speed_x_roadtype', 'first_road_class']
Numeric: ['hour', 'month', 'day_of_year', 'speed_limit']

collision_severity
Fatal        7550
Serious    116787
Slight     389361
Name: count, dtype: int64

collision_severity
Fatal       1.47
Serious    22.73
Slight     75.80
Name: proportion, dtype: float64


## 2. Stratified split

`stratify=y` keeps the Fatal / Serious / Slight mix the same in train and test. Without it, the test set could easily contain almost no Fatal rows, and recall on Fatal would be meaningless.

In [3]:
X_train, X_test, y_train, y_test = train.split_xy(X, y)
preprocessor = train.make_preprocessor(cat_cols, num_cols)

print("Train:", len(X_train), " Test:", len(X_test))
print("Train mix:\n", y_train.value_counts(normalize=True).sort_index().round(4).rename(config.SEVERITY_LABELS))
print("Test mix:\n", y_test.value_counts(normalize=True).sort_index().round(4).rename(config.SEVERITY_LABELS))

Train: 410958  Test: 102740
Train mix:
 collision_severity
Fatal      0.0147
Serious    0.2273
Slight     0.7580
Name: proportion, dtype: float64
Test mix:
 collision_severity
Fatal      0.0147
Serious    0.2273
Slight     0.7580
Name: proportion, dtype: float64


## 3. Dummy baseline — run this first

`DummyClassifier(strategy="most_frequent")` always predicts Slight.

Accuracy will look high because about 76% of collisions really are Slight. Macro F1 will be poor and **Fatal recall will be zero**. That is the Module 4 point: accuracy is the wrong metric on imbalanced data.

Screenshot this cell.

In [4]:
dummy = train.dummy_model()
dummy.fit(X_train, y_train)
dummy_result = train.evaluate(dummy, X_train, y_train, X_test, y_test, "dummy")


=== dummy ===
              precision    recall  f1-score   support

       Fatal       0.00      0.00      0.00      1510
     Serious       0.00      0.00      0.00     23357
      Slight       0.76      1.00      0.86     77873

    accuracy                           0.76    102740
   macro avg       0.25      0.33      0.29    102740
weighted avg       0.57      0.76      0.65    102740

accuracy=0.7580  macro_f1=0.2874  train_macro_f1=0.2874  test_macro_f1=0.2874


## 4. Logistic Regression — main model

`class_weight="balanced"` re-weights the loss so Fatal and Serious rows count as much, in total, as the Slight majority. Without it the model would mostly learn to say Slight.

Three classes are handled with a **multinomial** (softmax) Logistic Regression: one coefficient vector per class, probabilities that sum to 1.

`C` is the inverse regularisation strength. Smaller `C` = stronger penalty = simpler model. We sweep `C` on a stratified sample, then run 5-fold stratified CV on the full training set with the chosen value.

In [5]:
logreg, best_C, sweep = train.tune_logreg(preprocessor, X_train, y_train)
print("Best C:", best_C)
pd.DataFrame(sweep)

C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogReg C sweep (macro F1 on sample):
  C=0.1   mean macro F1=0.3150
  C=1.0   mean macro F1=0.3149
  C=10.0  mean macro F1=0.3149
Chosen C=0.1
Best C: 0.1


,Cs,mean_macro_f1,best_C
0,0.1,0.314954,0.1
1,1.0,0.314882,0.1
2,10.0,0.314901,0.1


In [6]:
print("5-fold stratified CV on the training set")
cv = train.cross_validate(logreg, X_train, y_train)
print(f"mean macro F1 = {cv['mean']:.4f}  std = {cv['std']:.4f}")

5-fold stratified CV on the training set


C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  fold 1: macro F1=0.3189


C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  fold 2: macro F1=0.3187


...

  fold 3: macro F1=0.3189


C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  fold 4: macro F1=0.3173


C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  fold 5: macro F1=0.3186
mean macro F1 = 0.3184  std = 0.0006


In [7]:
logreg.fit(X_train, y_train)
logreg_result = train.evaluate(logreg, X_train, y_train, X_test, y_test, "logistic_regression")

print()
print("Overfitting check: train macro F1 vs test macro F1")
print("train:", round(logreg_result["train_macro_f1"], 4))
print("test: ", round(logreg_result["test_macro_f1"], 4))
print("CV mean:", round(cv["mean"], 4))

C:\Users\Buddhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(



=== logistic_regression ===
              precision    recall  f1-score   support

       Fatal       0.04      0.65      0.07      1510
     Serious       0.25      0.25      0.25     23357
      Slight       0.81      0.54      0.64     77873

    accuracy                           0.47    102740
   macro avg       0.36      0.48      0.32    102740
weighted avg       0.67      0.47      0.55    102740

accuracy=0.4734  macro_f1=0.3206  train_macro_f1=0.3191  test_macro_f1=0.3206

Overfitting check: train macro F1 vs test macro F1
train: 0.3191
test:  0.3206
CV mean: 0.3184


If train and test macro F1 are close, the model is not memorising the training rows. Cross-validation being in the same range is the second check.

In [8]:
matrix = np.array(logreg_result["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=train.CLASS_NAMES,
    yticklabels=train.CLASS_NAMES,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Logistic Regression — confusion matrix")
fig.tight_layout()
fig.savefig(config.ARTIFACTS_DIR / "confusion_matrix.png", dpi=140)
plt.show()

C:\Users\Buddhi\AppData\Local\Temp\ipykernel_16004\2044826069.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Coefficient analysis

These weights are what Owner 2 turns into the "why is this risky" text. A positive KSI weight means that feature pushed the prediction towards Fatal or Serious relative to Slight.

In [9]:
coeffs = train.coefficient_table(logreg)
coeff_df = pd.DataFrame(coeffs)
print("Conditions that raise predicted severity")
display(coeff_df.head(15))
print()
print("Conditions that lower predicted severity")
display(coeff_df.tail(10))

Conditions that raise predicted severity


,feature,coef_fatal,coef_serious,coef_slight,ksi_weight
0,cat__road_type_6,0.382438,-0.033444,-0.348993,1.046980
1,cat__weather_conditions_4,0.350343,-0.025927,-0.324416,0.973249
2,num__speed_limit,0.300424,-0.101278,-0.199145,0.597436
3,cat__road_surface_conditions_2,0.236936,-0.039037,-0.197898,0.593695
4,cat__road_surface_conditions_1,0.209478,-0.030880,-0.178597,0.535792
5,cat__road_type_3,0.230732,-0.060884,-0.169848,0.509544
6,cat__time_of_day_night,0.175651,-0.020551,-0.155100,0.465299
7,cat__time_of_day_early_morning,0.226263,-0.073632,-0.152631,0.457893
8,cat__light_conditions_5,0.147837,-0.005779,-0.142058,0.426173
9,cat__junction_detail_0,0.224721,-0.085896,-0.138825,0.416474



Conditions that lower predicted severity


,feature,coef_fatal,coef_serious,coef_slight,ksi_weight
46,cat__urban_or_rural_area_1,-0.217655,0.050924,0.166731,-0.500192
47,cat__light_conditions_1,-0.255451,0.081109,0.174342,-0.523027
48,cat__road_type_1,-0.348566,0.155322,0.193244,-0.579733
49,cat__weather_conditions_8,-0.272517,0.074627,0.197891,-0.593672
50,cat__time_of_day_morning_rush,-0.258345,0.031622,0.226724,-0.680171
51,cat__junction_detail_99,-0.145047,-0.087134,0.232181,-0.696542
52,cat__first_road_class_1,-0.233472,-0.010365,0.243836,-0.731509
53,cat__weather_conditions_99,-0.294076,-0.003735,0.297811,-0.893433
54,cat__road_type_99,-0.354510,-0.052826,0.407336,-1.222009
55,cat__road_surface_conditions_99,-0.471671,0.022017,0.449654,-1.348962


## 6. Comparison models

Random Forest and XGBoost are **beyond the module**. They are here to ask one question: does a non-linear model help enough to justify losing readable coefficients?

They are not the served model.

In [10]:
forest = train.forest_pipeline(preprocessor)
forest.fit(X_train, y_train)
forest_result = train.evaluate(forest, X_train, y_train, X_test, y_test, "random_forest")


=== random_forest ===
              precision    recall  f1-score   support

       Fatal       0.04      0.54      0.08      1510
     Serious       0.27      0.24      0.25     23357
      Slight       0.80      0.64      0.71     77873

    accuracy                           0.55    102740
   macro avg       0.37      0.47      0.35    102740
weighted avg       0.67      0.55      0.60    102740

accuracy=0.5475  macro_f1=0.3465  train_macro_f1=0.3714  test_macro_f1=0.3465


In [11]:
xgb = train.xgb_pipeline(preprocessor)
train.fit_xgboost(xgb, X_train, y_train)
xgb_result = train.evaluate(xgb, X_train, y_train, X_test, y_test, "xgboost")


=== xgboost ===
              precision    recall  f1-score   support

       Fatal       0.04      0.62      0.07      1510
     Serious       0.25      0.26      0.26     23357
      Slight       0.81      0.56      0.66     77873

    accuracy                           0.49    102740
   macro avg       0.37      0.48      0.33    102740
weighted avg       0.67      0.49      0.56    102740

accuracy=0.4910  macro_f1=0.3292  train_macro_f1=0.3373  test_macro_f1=0.3292


## 7. Compare and choose

We do not pick the highest accuracy. We look at macro F1, Fatal/Serious recall, whether we can explain the prediction, and whether the output is a usable probability.

In [12]:
comparison = pd.DataFrame([
    train.comparison_row("dummy", dummy_result),
    train.comparison_row("logistic_regression", logreg_result),
    train.comparison_row("random_forest", forest_result),
    train.comparison_row("xgboost", xgb_result),
])
display(comparison.round(4))

,model,accuracy,macro_precision,macro_recall,macro_f1
0,dummy,0.7580,0.2527,0.3333,0.2874
1,logistic_regression,0.4734,0.3636,0.4792,0.3206
2,random_forest,0.5475,0.3692,0.4712,0.3465
3,xgboost,0.4910,0.3662,0.4792,0.3292


**Choice: Logistic Regression (`logreg-v1`).**

The dummy baseline proves accuracy is misleading. Logistic Regression, with `class_weight="balanced"`, is the first model that actually looks at Fatal and Serious cases.

Tree models may score a little higher on macro F1. We still serve Logistic Regression because:

1. Its coefficients become the in-app explanations. A safety tool that cannot say *why* a stretch is risky will not be trusted.
2. `predict_proba` is well-behaved and is what the scoring formula uses.
3. The module taught this model. The comparison models are a check, not a replacement.

That is the production decision. The backend loads this pipeline once at startup.

## 8. Save the whole pipeline

`model.joblib` contains the preprocessor **and** the model. `metrics.json` contains the feature column list the backend checks on startup.

In [13]:
(config.ARTIFACTS_DIR / "coefficients.json").write_text(
    __import__("json").dumps(coeffs[:40], indent=2)
)

metrics = {
    "chosen_model": train.CHOSEN_MODEL,
    "selection_reason": (
        "Random Forest and XGBoost are included as a check, not as the served "
        "model. Logistic Regression is the production choice because its "
        "coefficients can be turned into readable reasons for each segment, "
        "class_weight='balanced' lifts recall on Fatal/Serious, and the "
        "backend already scores from those coefficients."
    ),
    "feature_columns": feature_columns,
    "class_labels": train.CLASS_NAMES,
    "best_C": best_C,
    "C_sweep": sweep,
    "cv_macro_f1_mean": cv["mean"],
    "cv_macro_f1_std": cv["std"],
    "cv_folds": cv["folds"],
    "train_macro_f1": logreg_result["train_macro_f1"],
    "test_macro_f1": logreg_result["test_macro_f1"],
    "accuracy": logreg_result["accuracy"],
    "macro_precision": logreg_result["macro_precision"],
    "macro_recall": logreg_result["macro_recall"],
    "macro_f1": logreg_result["macro_f1"],
    "per_class": logreg_result["per_class"],
    "confusion_matrix": logreg_result["confusion_matrix"],
    "comparison": comparison.to_dict(orient="records"),
    "top_ksi_coefficients": coeffs[:15],
}

print("Saved", train.save_metrics(metrics))
print("Saved", train.save_pipeline(logreg))
print("Owner 1 can now load model.joblib. /api/health should show model_loaded: true.")

Saved D:\nethushi\roadsafe-ai\ml\artifacts\metrics.json
Saved D:\nethushi\roadsafe-ai\ml\artifacts\model.joblib
Owner 1 can now load model.joblib. /api/health should show model_loaded: true.
